# Laser Off Code

In [ ]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

# Import

In [1]:
import time
from time import sleep, monotonic
import datetime
import numpy as np
import matplotlib.pyplot as plt
import sys
import pyvisa
import qcodes as qc
from qcodes.dataset import Measurement
from qcodes.dataset import do0d
from qcodes.dataset.experiment_container import new_experiment, load_experiment_by_name
from qcodes.dataset.plotting import plot_by_id
from qcodes.dataset.data_set import load_by_id, load_by_counter
from qcodes import initialise_or_create_database_at, new_data_set, new_experiment
from qcodes.station import Station
initialise_or_create_database_at("./2026-04-17_SNSPD7.db")
from functions import quick_check
from functions import calibrate
import snspd
params = snspd.snspd('D:\SNSPD\SNSPD2\SNSPD5\snspd6-1.yaml')

# Set up experiment
exp_name = 'SNSPD7'
sample_name = '00'

try:
    exp = qc.load_experiment_by_name(exp_name, sample=sample_name)
    print('Experiment loaded. Last ID no:', exp.last_counter)
except ValueError:
    exp = new_experiment(exp_name, sample_name)
    print('Started new experiment')

<>:20: SyntaxWarning: invalid escape sequence '\S'
<>:20: SyntaxWarning: invalid escape sequence '\S'
C:\Users\QNL\AppData\Local\Temp\ipykernel_39124\2061385194.py:20: SyntaxWarning: invalid escape sequence '\S'
  params = snspd.snspd('D:\SNSPD\SNSPD2\SNSPD5\snspd6-1.yaml')


Logging hadn't been started.
Activating auto-logging. Current session state plus future input saved.
Filename       : C:\Users\QNL\.qcodes\logs\command_history.log
Mode           : append
Output logging : True
Raw input log  : False
Timestamping   : True
State          : active
Qcodes Logfile : C:\Users\QNL\.qcodes\logs\260424-39124-qcodes.log
Started new experiment


In [2]:
station = Station(config_file="friesland.yaml")

In [4]:
laser = station.load_instrument("laser", revive_instance=True)

Connected to: PurePhotonic PPCL550 (serial:PP70AJ005, firmware:PV 2.0.0:HW 3.0.0:FW 7.0.0:AS C1:OT 1.0.0) in 1.58s


In [5]:
pm100d = station.load_instrument("pm100d", revive_instance=True)

2026-04-24 15:07:42,357 ¦ qcodes.instrument.instrument_base ¦ WARNING ¦ instrument_base ¦ snapshot_base ¦ 464 ¦ [pm100d(Thorlabs_PM100D)] Snapshot: Could not update parameter: wavelength


Connected to: Thorlabs PM100D (serial:P0033329, firmware:2.8.1) in 5.61s


2026-04-24 15:07:42,604 ¦ qcodes.instrument.instrument_base ¦ WARNING ¦ instrument_base ¦ snapshot_base ¦ 464 ¦ [pm100d(Thorlabs_PM100D)] Snapshot: Could not update parameter: beam_diameter


In [6]:
pms120 = station.load_instrument("pms120", revive_instance=True)

2026-04-24 15:07:42,768 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



In [7]:
tc = station.load_instrument("fridge", revive_instance=True)

In [6]:
# import pyvisa 
# rm = pyvisa.ResourceManager()
# rm.list_resources()
# my_instrument = rm.open_resource('USB0::0x1313::0x8072::1906768::0::INSTR')

('TCPIP0::10.196.52.98::inst0::INSTR',
 'ASRL1::INSTR',
 'ASRL2::INSTR',
 'ASRL5::INSTR',
 'ASRL6::INSTR',
 'ASRL7::INSTR',
 'ASRL8::INSTR',
 'ASRL9::INSTR',
 'ASRL10::INSTR',
 'ASRL11::INSTR',
 'ASRL12::INSTR',
 'ASRL13::INSTR',
 'TCPIP0::10.196.50.27::inst0::INSTR',
 'USB0::0x05E6::0x2230::9010428::0::INSTR',
 'USB0::0x1313::0x8072::1906768::0::INSTR',
 'USB0::0x1313::0x8072::1913782::0::INSTR',
 'USB0::0x1313::0x8078::P0033329::0::INSTR')

# Calibration

Fridge fibre only. Single fibre connected. 
Laser -> beam splitter -> 10% port to fridge -> PM100D
Room temperature, Still thermometer above dew point this is what we waited for to open the fridge 

Connection: through optical breakout box 

In [9]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

power10, power90, attenuation = quick_check(bs10=params.bs10, bs90=params.bs90, pmeter10=pm100d, pmeter90=pms120, attenuator_name = params.connection1)

print(f"Power on PM100USB after attenuator is:{power10}\nPower on PM120 after 90% beamsplitter port is:{power90}\nAttenuation is:{attenuation}\n")

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True
Starting experimental run with id: 1. 
1


2026-04-24 15:08:35,798 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Power on PM100USB after attenuator is:0.000297434075
Power on PM120 after 90% beamsplitter port is:0.00488496805
Attenuation is:2.0099730614843576

Laser enable status: False


Connection: through feedthrough

In [10]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

power10, power90, attenuation = quick_check(bs10=params.bs10, bs90=params.bs90, pmeter10=pm100d, pmeter90=pms120, attenuator_name = params.connection1)

print(f"Power on PM100USB after attenuator is:{power10}\nPower on PM120 after 90% beamsplitter port is:{power90}\nAttenuation is:{attenuation}\n")

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True
Starting experimental run with id: 2. 
2


2026-04-24 15:19:16,511 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Power on PM100USB after attenuator is:0.000384206738
Power on PM120 after 90% beamsplitter port is:0.00489019789
Attenuation is:0.9028775670375186

Laser enable status: False


In [11]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

power10, power90, attenuation = quick_check(bs10=params.bs10, bs90=params.bs90, pmeter10=pm100d, pmeter90=pms120, attenuator_name = params.connection1)

print(f"Power on PM100USB after attenuator is:{power10}\nPower on PM120 after 90% beamsplitter port is:{power90}\nAttenuation is:{attenuation}\n")

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True
Starting experimental run with id: 3. 
3


2026-04-24 15:26:51,850 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Power on PM100USB after attenuator is:0.000381087535
Power on PM120 after 90% beamsplitter port is:0.00484941667
Attenuation is:0.9019105714852991

Laser enable status: False


In [12]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

power10, power90, attenuation = quick_check(bs10=params.bs10, bs90=params.bs90, pmeter10=pm100d, pmeter90=pms120, attenuator_name = params.connection1)

print(f"Power on PM100USB after attenuator is:{power10}\nPower on PM120 after 90% beamsplitter port is:{power90}\nAttenuation is:{attenuation}\n")

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True
Starting experimental run with id: 4. 
4


2026-04-24 15:50:11,205 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Power on PM100USB after attenuator is:0.000381557213
Power on PM120 after 90% beamsplitter port is:0.00485778181
Attenuation is:0.9040463618530937

Laser enable status: False


In [13]:
from datetime import datetime
meas = Measurement()
meas.register_custom_parameter("power10", label="W")
meas.register_custom_parameter("power90", label="W")
meas.register_custom_parameter("attenuation", label="dB")
meas.register_custom_parameter("still_temp", label="K")
meas.register_custom_parameter("fourK_temp", label="K")
meas.register_custom_parameter("fiftyK_temp", label="K")


bs10=params.bs10
bs90=params.bs90
pmeter10=pm100d
pmeter90=pms120
attenuator_name = params.connection1

############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)


with meas.run() as datasaver:
    print(datasaver.run_id)
    datasaver.dataset.add_metadata("attenuator_name", attenuator_name)

    still_temp = tc.Still_temp()
    print(still_temp)
    
    while True:

        if tc.Still_temp() < (still_temp - 0.5):
            power10 = pmeter10.power()
            power90 = pmeter90.power()
            attenuation = 10*np.log10((bs10/bs90*power90)/power10)
            time.sleep(2)
        
            still_temp = tc.Still_temp()
            datasaver.add_result(("power10", power10),
                    ("power90", power90),
                    ("attenuation", attenuation),
                    ("fiftyK_temp", tc.Fifty_K_temp()),
                    ("fourK_temp", tc.Four_K_temp()),
                    ("still_temp", still_temp))
            
            print(f"Power on PM100USB after attenuator is:{power10}\nPower on PM120 after 90% beamsplitter port is:{power90}\nAttenuation is:{attenuation}\n")
            print(f'1K Temp: {still_temp}')

        print(f'Starting sleep {datetime.now().time()}')
        time.sleep(30)

Laser enable status: True
Starting experimental run with id: 5. 
5
263.4963
Starting sleep 19:09:47.537597
Starting sleep 19:10:20.574999
Starting sleep 19:10:53.624092


2026-04-24 19:11:26,879 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Power on PM100USB after attenuator is:0.000373632676
Power on PM120 after 90% beamsplitter port is:0.00488601346
Attenuation is:1.0203610842274156

1K Temp: 262.87
Starting sleep 19:11:35.026020
Starting sleep 19:12:08.086730
Starting sleep 19:12:41.129083
Starting sleep 19:13:11.176731
Starting sleep 19:13:44.217443
Power on PM100USB after attenuator is:0.000372464478
Power on PM120 after 90% beamsplitter port is:0.00487974007
Attenuation is:1.0283812843259927

1K Temp: 262.0804
Starting sleep 19:14:23.926324
Starting sleep 19:14:56.970783
Starting sleep 19:15:30.010810
Starting sleep 19:16:02.548858
Starting sleep 19:16:35.593584
Power on PM100USB after attenuator is:0.000370645896
Power on PM120 after 90% beamsplitter port is:0.00487346621
Attenuation is:1.0440506329288848

1K Temp: 261.4678
Starting sleep 19:17:20.683887
Starting sleep 19:17:53.733218
Starting sleep 19:18:26.769108
Starting sleep 19:18:57.593248
Power on PM100USB after attenuator is:0.00037042913
Power on PM120 aft

2026-04-26 01:21:48,538 ¦ qcodes.dataset.measurements ¦ WARNING ¦ measurements ¦ __exit__ ¦ 758 ¦ An exception occurred in measurement with guid: 0a5068be-0000-0000-0000-019dbec08456;
Traceback:
Traceback (most recent call last):
  File "C:\Users\QNL\AppData\Local\Temp\ipykernel_39124\3687984759.py", line 50, in <module>
    time.sleep(30)
    ~~~~~~~~~~^^^^
KeyboardInterrupt



KeyboardInterrupt: 

In [14]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: False


In [15]:
print(f'50K Temp: {tc.Fifty_K_temp()}')
print(f'4K Temp: {tc.Four_K_temp()}')
print(f'Still Temp: {tc.Still_temp()}')
print(f'MXC Temp: {tc.MC_temp()}')

50K Temp: 42.61996
4K Temp: 3.054668
Still Temp: 3.610557
MXC Temp: 2.994991


In [16]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

power10, power90, attenuation = quick_check(bs10=params.bs10, bs90=params.bs90, pmeter10=pm100d, pmeter90=pms120, attenuator_name = params.connection1)

print(f"Power on PM100USB after attenuator is:{power10}\nPower on PM120 after 90% beamsplitter port is:{power90}\nAttenuation is:{attenuation}\n")

print(f'{datetime.now()}')
print(f'50K Temp: {tc.Fifty_K_temp()}')
print(f'4K Temp: {tc.Four_K_temp()}')
print(f'Still Temp: {tc.Still_temp()}')
print(f'MXC Temp: {tc.MC_temp()}')

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True
Starting experimental run with id: 6. 
6


2026-04-26 21:06:08,060 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Power on PM100USB after attenuator is:5.15432885e-06
Power on PM120 after 90% beamsplitter port is:0.00487660291
Attenuation is:19.614715698497918

2026-04-26 21:06:10.206464
50K Temp: 42.61996
4K Temp: 3.055244
Still Temp: 3.610419
MXC Temp: 2.997818
Laser enable status: False


In [17]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

power10, power90, attenuation = quick_check(bs10=params.bs10, bs90=params.bs90, pmeter10=pm100d, pmeter90=pms120, attenuator_name = params.connection1)

print(f"Power on PM100USB after attenuator is:{power10}\nPower on PM120 after 90% beamsplitter port is:{power90}\nAttenuation is:{attenuation}\n")

print(f'{datetime.now()}')
print(f'50K Temp: {tc.Fifty_K_temp()}')
print(f'4K Temp: {tc.Four_K_temp()}')
print(f'Still Temp: {tc.Still_temp()}')
print(f'MXC Temp: {tc.MC_temp()}')

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True
Starting experimental run with id: 7. 
7


2026-04-26 21:06:51,167 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Power on PM100USB after attenuator is:5.20487492e-06
Power on PM120 after 90% beamsplitter port is:0.00487451162
Attenuation is:19.570471124857875

2026-04-26 21:06:53.302546
50K Temp: 42.61996
4K Temp: 3.055244
Still Temp: 3.610419
MXC Temp: 3.001525
Laser enable status: False


In [18]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

power10, power90, attenuation = quick_check(bs10=params.bs10, bs90=params.bs90, pmeter10=pm100d, pmeter90=pms120, attenuator_name = params.connection1)

print(f"Power on PM100USB after attenuator is:{power10}\nPower on PM120 after 90% beamsplitter port is:{power90}\nAttenuation is:{attenuation}\n")

print(f'{datetime.now()}')
print(f'50K Temp: {tc.Fifty_K_temp()}')
print(f'4K Temp: {tc.Four_K_temp()}')
print(f'Still Temp: {tc.Still_temp()}')
print(f'MXC Temp: {tc.MC_temp()}')

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True
Starting experimental run with id: 8. 
8


2026-04-26 21:07:30,926 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Power on PM100USB after attenuator is:5.20051117e-06
Power on PM120 after 90% beamsplitter port is:0.00486928364
Attenuation is:19.569453395721958

2026-04-26 21:07:33.090041
50K Temp: 42.62306
4K Temp: 3.055508
Still Temp: 3.610281
MXC Temp: 3.001525
Laser enable status: False


In [19]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: False


Note: don't turn laser off between these next time 

# OLD

In [11]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

power10, power90, attenuation = quick_check(bs10=params.bs10, bs90=params.bs90, pmeter10=pm100d, pmeter90=pms120, attenuator_name = params.connection1)

print(f"Power on PM100USB after attenuator is:{power10}\nPower on PM120 after 90% beamsplitter port is:{power90}\nAttenuation is:{attenuation}\n")

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True
Starting experimental run with id: 50. 
50


2026-04-21 17:46:33,665 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Power on PM100USB after attenuator is:0.000328939612
Power on PM120 after 90% beamsplitter port is:0.00486196438
Attenuation is:1.5522189672907338

Laser enable status: False


In [12]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

power10, power90, attenuation = quick_check(bs10=params.bs10, bs90=params.bs90, pmeter10=pm100d, pmeter90=pms120, attenuator_name = params.connection1)

print(f"Power on PM100USB after attenuator is:{power10}\nPower on PM120 after 90% beamsplitter port is:{power90}\nAttenuation is:{attenuation}\n")

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True
Starting experimental run with id: 51. 
51


2026-04-21 17:47:36,763 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Power on PM100USB after attenuator is:0.000329493632
Power on PM120 after 90% beamsplitter port is:0.00486196438
Attenuation is:1.5449104697979568

Laser enable status: False


Room temperature fridge closed, pumping/cooling started

In [13]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

power10, power90, attenuation = quick_check(bs10=params.bs10, bs90=params.bs90, pmeter10=pm100d, pmeter90=pms120, attenuator_name = params.connection1)

print(f"Power on PM100USB after attenuator is:{power10}\nPower on PM120 after 90% beamsplitter port is:{power90}\nAttenuation is:{attenuation}\n")

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True
Starting experimental run with id: 52. 
52


2026-04-22 09:40:21,150 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Power on PM100USB after attenuator is:0.00032896371
Power on PM120 after 90% beamsplitter port is:0.00486614695
Attenuation is:1.5556352867229764

Laser enable status: False


While cooling MXC above range, 50K flange measuring 164.73K 

In [14]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

power10, power90, attenuation = quick_check(bs10=params.bs10, bs90=params.bs90, pmeter10=pm100d, pmeter90=pms120, attenuator_name = params.connection1)

print(f"Power on PM100USB after attenuator is:{power10}\nPower on PM120 after 90% beamsplitter port is:{power90}\nAttenuation is:{attenuation}\n")

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True
Starting experimental run with id: 53. 
53


2026-04-22 13:29:15,102 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Power on PM100USB after attenuator is:0.000311332202
Power on PM120 after 90% beamsplitter port is:0.00485673593
Attenuation is:1.7864674711945117

Laser enable status: False


50K Flange at 118.35K

In [15]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

power10, power90, attenuation = quick_check(bs10=params.bs10, bs90=params.bs90, pmeter10=pm100d, pmeter90=pms120, attenuator_name = params.connection1)

print(f"Power on PM100USB after attenuator is:{power10}\nPower on PM120 after 90% beamsplitter port is:{power90}\nAttenuation is:{attenuation}\n")

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True
Starting experimental run with id: 54. 
54


2026-04-22 15:12:57,416 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Power on PM100USB after attenuator is:0.000247911667
Power on PM120 after 90% beamsplitter port is:0.00485255383
Attenuation is:2.7719969591721223

Laser enable status: False


In [19]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

power10, power90, attenuation = quick_check(bs10=params.bs10, bs90=params.bs90, pmeter10=pm100d, pmeter90=pms120, attenuator_name = params.connection1)

print(f"Power on PM100USB after attenuator is:{power10}\nPower on PM120 after 90% beamsplitter port is:{power90}\nAttenuation is:{attenuation}\n")

print(f'50K Temp: {tc.Fifty_K_temp()}')
print(f'4K Temp: {tc.Four_K_temp()}')
print(f'1K Temp: {tc.Still_temp()}')

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True
Starting experimental run with id: 55. 
55


2026-04-22 15:15:46,731 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Power on PM100USB after attenuator is:0.000245804054
Power on PM120 after 90% beamsplitter port is:0.00486510107
Attenuation is:2.8202912537651104

50K Temp: 117.5981
4K Temp: 170.6838
1K Temp: 239.807
Laser enable status: False


In [20]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

power10, power90, attenuation = quick_check(bs10=params.bs10, bs90=params.bs90, pmeter10=pm100d, pmeter90=pms120, attenuator_name = params.connection1)

print(f"Power on PM100USB after attenuator is:{power10}\nPower on PM120 after 90% beamsplitter port is:{power90}\nAttenuation is:{attenuation}\n")

print(f'50K Temp: {tc.Fifty_K_temp()}')
print(f'4K Temp: {tc.Four_K_temp()}')
print(f'1K Temp: {tc.Still_temp()}')

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True
Starting experimental run with id: 56. 
56


2026-04-22 16:54:59,116 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Power on PM100USB after attenuator is:0.000191175248
Power on PM120 after 90% beamsplitter port is:0.00483582215
Attenuation is:3.8856495770068507

50K Temp: 92.64624
4K Temp: 143.4087
1K Temp: 215.6713
Laser enable status: False


In [21]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

power10, power90, attenuation = quick_check(bs10=params.bs10, bs90=params.bs90, pmeter10=pm100d, pmeter90=pms120, attenuator_name = params.connection1)

print(f"Power on PM100USB after attenuator is:{power10}\nPower on PM120 after 90% beamsplitter port is:{power90}\nAttenuation is:{attenuation}\n")

print(f'50K Temp: {tc.Fifty_K_temp()}')
print(f'4K Temp: {tc.Four_K_temp()}')
print(f'1K Temp: {tc.Still_temp()}')

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True
Starting experimental run with id: 57. 
57


2026-04-23 08:35:13,973 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Power on PM100USB after attenuator is:1.00159723e-05
Power on PM120 after 90% beamsplitter port is:0.00487137493
Attenuation is:16.72484741733037

50K Temp: 55.44603
4K Temp: 23.40625
1K Temp: 59.66102
Laser enable status: False


In [22]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

power10, power90, attenuation = quick_check(bs10=params.bs10, bs90=params.bs90, pmeter10=pm100d, pmeter90=pms120, attenuator_name = params.connection1)

print(f"Power on PM100USB after attenuator is:{power10}\nPower on PM120 after 90% beamsplitter port is:{power90}\nAttenuation is:{attenuation}\n")

print(f'50K Temp: {tc.Fifty_K_temp()}')
print(f'4K Temp: {tc.Four_K_temp()}')
print(f'1K Temp: {tc.Still_temp()}')

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True
Starting experimental run with id: 58. 
58


2026-04-23 11:46:00,616 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Power on PM100USB after attenuator is:5.0265694e-06
Power on PM120 after 90% beamsplitter port is:0.00485359924
Attenuation is:19.703185306332365

50K Temp: 54.77798
4K Temp: 16.47031
1K Temp: 42.36339
Laser enable status: False


In [23]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

power10, power90, attenuation = quick_check(bs10=params.bs10, bs90=params.bs90, pmeter10=pm100d, pmeter90=pms120, attenuator_name = params.connection1)

print(f"Power on PM100USB after attenuator is:{power10}\nPower on PM120 after 90% beamsplitter port is:{power90}\nAttenuation is:{attenuation}\n")

print(f'50K Temp: {tc.Fifty_K_temp()}')
print(f'4K Temp: {tc.Four_K_temp()}')
print(f'1K Temp: {tc.Still_temp()}')

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True
Starting experimental run with id: 59. 
59


2026-04-23 12:50:38,096 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Power on PM100USB after attenuator is:4.01479747e-06
Power on PM120 after 90% beamsplitter port is:0.0048452341
Attenuation is:20.67177423036082

50K Temp: 53.79736
4K Temp: 13.63159
1K Temp: 36.02
Laser enable status: False


In [25]:
plaser = 0.0048452341/params.bs90
pin = plaser*params.bs10
pout = 4.01479747e-06

In [30]:
attenuation = 10*np.log(pout/pin)

In [31]:
attenuation

np.float64(-47.59851918856729)

In [32]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

power10, power90, attenuation = quick_check(bs10=params.bs10, bs90=params.bs90, pmeter10=pm100d, pmeter90=pms120, attenuator_name = params.connection1)

print(f"Power on PM100USB after attenuator is:{power10}\nPower on PM120 after 90% beamsplitter port is:{power90}\nAttenuation is:{attenuation}\n")

print(f'50K Temp: {tc.Fifty_K_temp()}')
print(f'4K Temp: {tc.Four_K_temp()}')
print(f'1K Temp: {tc.Still_temp()}')

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True
Starting experimental run with id: 60. 
60


2026-04-23 18:04:33,846 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Power on PM100USB after attenuator is:1.2889401e-06
Power on PM120 after 90% beamsplitter port is:0.00485882722
Attenuation is:25.618250204320226

50K Temp: 43.90189
4K Temp: 3.338257
1K Temp: 4.861306
Laser enable status: False


In [33]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

power10, power90, attenuation = quick_check(bs10=params.bs10, bs90=params.bs90, pmeter10=pm100d, pmeter90=pms120, attenuator_name = params.connection1)

print(f"Power on PM100USB after attenuator is:{power10}\nPower on PM120 after 90% beamsplitter port is:{power90}\nAttenuation is:{attenuation}\n")

print(f'50K Temp: {tc.Fifty_K_temp()}')
print(f'4K Temp: {tc.Four_K_temp()}')
print(f'1K Temp: {tc.Still_temp()}')

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True
Starting experimental run with id: 61. 
61


2026-04-23 18:06:53,477 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Power on PM100USB after attenuator is:1.28869772e-06
Power on PM120 after 90% beamsplitter port is:0.00483686756
Attenuation is:25.599394356038765

50K Temp: 43.88377
4K Temp: 3.334722
1K Temp: 4.842457
Laser enable status: False


In [34]:
############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)

power10, power90, attenuation = quick_check(bs10=params.bs10, bs90=params.bs90, pmeter10=pm100d, pmeter90=pms120, attenuator_name = params.connection1)

print(f"Power on PM100USB after attenuator is:{power10}\nPower on PM120 after 90% beamsplitter port is:{power90}\nAttenuation is:{attenuation}\n")

print(f'50K Temp: {tc.Fifty_K_temp()}')
print(f'4K Temp: {tc.Four_K_temp()}')
print(f'1K Temp: {tc.Still_temp()}')

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: True
Starting experimental run with id: 62. 
62


2026-04-23 18:07:32,462 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Power on PM100USB after attenuator is:1.29536443e-06
Power on PM120 after 90% beamsplitter port is:0.00486614695
Attenuation is:25.603195521903945

50K Temp: 43.87472
4K Temp: 3.333115
1K Temp: 4.842457
Laser enable status: False


In [41]:
from datetime import datetime
meas = Measurement()
meas.register_custom_parameter("power10", label="W")
meas.register_custom_parameter("power90", label="W")
meas.register_custom_parameter("attenuation", label="dB")
meas.register_custom_parameter("still_temp", label="K")
meas.register_custom_parameter("fourK_temp", label="K")
meas.register_custom_parameter("fiftyK_temp", label="K")


bs10=params.bs10
bs90=params.bs90
pmeter10=pm100d
pmeter90=pms120
attenuator_name = params.connection1

############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)


with meas.run() as datasaver:
    print(datasaver.run_id)
    datasaver.dataset.add_metadata("attenuator_name", attenuator_name)

    still_temp = tc.Still_temp()
    print(still_temp)
    
    while True:

        if tc.Still_temp() > (still_temp + 0.5):
            power10 = pmeter10.power()
            power90 = pmeter90.power()
            attenuation = 10*np.log10((bs10/bs90*power90)/power10)
            time.sleep(2)
        
            still_temp = tc.Still_temp()
            datasaver.add_result(("power10", power10),
                    ("power90", power90),
                    ("attenuation", attenuation),
                    ("fiftyK_temp", tc.Fifty_K_temp()),
                    ("fourK_temp", tc.Four_K_temp()),
                    ("still_temp", still_temp))
            
            print(f"Power on PM100USB after attenuator is:{power10}\nPower on PM120 after 90% beamsplitter port is:{power90}\nAttenuation is:{attenuation}\n")
            print(f'1K Temp: {still_temp}')

        print(f'Starting sleep {datetime.now().time()}')
        time.sleep(30)

Laser enable status: True
Starting experimental run with id: 69. 
69
6.791499
Starting sleep 18:48:32.151693
Starting sleep 18:49:05.188861
Starting sleep 18:49:38.241135


2026-04-23 18:50:11,491 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Power on PM100USB after attenuator is:1.54385282e-06
Power on PM120 after 90% beamsplitter port is:0.00486196438
Attenuation is:24.837321771328945

1K Temp: 7.291614
Starting sleep 18:50:22.651166
Starting sleep 18:50:55.694946
Starting sleep 18:51:28.738963
Power on PM100USB after attenuator is:1.53645874e-06
Power on PM120 after 90% beamsplitter port is:0.00487242034
Attenuation is:24.867501450512716

1K Temp: 7.86578
Starting sleep 18:52:13.173903
Starting sleep 18:52:46.222306
Starting sleep 18:53:19.282256
Starting sleep 18:53:52.323792
Power on PM100USB after attenuator is:1.53730741e-06
Power on PM120 after 90% beamsplitter port is:0.0048755575
Attenuation is:24.867898619081465

1K Temp: 8.439036
Starting sleep 18:54:36.735009
Starting sleep 18:55:09.785448
Starting sleep 18:55:42.842206
Starting sleep 18:56:15.886712
Starting sleep 18:56:48.952776
Power on PM100USB after attenuator is:1.53839824e-06
Power on PM120 after 90% beamsplitter port is:0.00487242034
Attenuation is:24.8

2026-04-24 14:12:19,098 ¦ qcodes.dataset.measurements ¦ WARNING ¦ measurements ¦ __exit__ ¦ 758 ¦ An exception occurred in measurement with guid: 8ebcd717-0000-0000-0000-019db986b288;
Traceback:
Traceback (most recent call last):
  File "C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\urllib3\connection.py", line 204, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\urllib3\util\connection.py", line 85, in create_connection
    raise err
  File "C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\urllib3\util\connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
TimeoutError: [WinError 10060] A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond

The

ConnectTimeout: (MaxRetryError("HTTPSConnectionPool(host='qsyd.sydney.edu.au', port=443): Max retries exceeded with url: /data/Friesland?current (Caused by ConnectTimeoutError(<HTTPSConnection(host='qsyd.sydney.edu.au', port=443) at 0x1b8dff3e5d0>, 'Connection to qsyd.sydney.edu.au timed out. (connect timeout=None)'))"), 'getting fridge_Still_temp')

In [42]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Laser enable status: False
